# IVADO Bootcamp 2026 — Labs 2 & 3: nano-R1 LLM RL

**Goal.** Reuse the policy-gradient machinery from Lab 1 and apply it to
a small language model. We'll train **Qwen2.5-3B-Base** on the Countdown
arithmetic task with **GRPO** — a group-relative variant of REINFORCE —
and watch the same emergent reasoning behaviors that DeepSeek R1-Zero
showed at scale: self-reflection, verification, backtracking. The loss
form is exactly the one you implemented this morning; only the alphabet
changes (gridworld states → text tokens).

**Requirements.** A100-80GB Colab runtime (*Runtime → Change runtime
type → A100*). The 3B policy + reference model + vLLM engine peak at
~57 GB of GPU memory.

## Table of contents

§0 Colab setup · §1 Imports & DeepSpeed env · §2 Hyperparameters ·
§3 Prompts & Countdown dataset · §4 STUDENT: reward functions ·
§5 STUDENT: episode generation · §6 STUDENT: policy-gradient loss ·
§7 Init policy + reference + vLLM + wandb · §8 Training loop

---

Original tutorial: <https://github.com/McGill-NLP/nano-aha-moment>


## 0. Colab setup

Run the cells in this section once at the start of a fresh runtime.

In [ ]:
# Confirm an A100-80GB is attached.
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Clone the repo and cd into it.
%cd /content
![ -d nano-aha-moment ] || git clone https://github.com/McGill-NLP/nano-aha-moment.git
%cd nano-aha-moment


In [ ]:
# Install flash-attn from the prebuilt wheel that matches Colab's Python 3.11 + torch 2.6 + cu12.
# Using the wheel avoids a 10+ minute on-VM build.
!pip install -q packaging wheel ninja
!pip install -q https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl


In [ ]:
# Install the rest of the requirements (torch 2.6 + flash-attn already present).
!grep -v -E '^(flash-attn|torch==)' requirements.txt > /tmp/req_filtered.txt
!pip install -q -r /tmp/req_filtered.txt
!pip install -q ipytest


In [ ]:
# Environment for this notebook.
import os
os.environ['WANDB_MODE'] = 'disabled'   # no W&B account needed
os.environ['VLLM_USE_V1'] = '0'         # vLLM 0.8.5 needs the V0 engine for sleep-mode
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.chdir('/content/nano-aha-moment')
print('CWD:', os.getcwd())


R1-Zero is arguably the more interesting contribution from the DeepSeek R1 paper. The core idea: take a freshly pre-trained LLM (straight out of the unsupervised pretraining oven) and continue its training using reinforcement learning *without* any human feedback or supervision. The result? A model that starts showing emergent behaviors like self-reflection, verification, backtracking that researchers have tried to bake into LLMs using handcrafted tricks and inductive biases, at least since O1.

In this notebook, we’ll build an R1-Zero-style training loop **from scratch**. The goal is to create a crystal-clear, hackable foundation for RL-style LLM training; one that gives you a bird’s-eye view of every moving part and how they fit together. Perfect for playing around, extending, or hacking.

---

### Why another R1-Zero implementation?

There are already great implementations like [TinyZero](https://github.com/Jiayi-Pan/TinyZero) and [Mini-R1](https://www.philschmid.de/mini-deepseek-r1). But they rely on full-fledged RL libraries (like `trl` or `verl`) to handle training.

These libraries exist for good reason; efficient RL training for LLMs sits at the crossroads of scalable training and fast inference. Making that work takes a lot of engineering. But that also means the internals are often abstracted away, hard to read, and even harder to tweak.

This notebook is different: **no abstractions, no hiding**. You’ll see everything, top to bottom. A lightweight, readable codebase that still follows best practices and runs efficiently on a single GPU.

### What is this notebook, exactly?

We'll train a base LLM using RL to solve a reasoning-heavy algorithmic task. The setup:

- **Model**: Qwen2.5 3B-Base  
- **Dataset**: Countdown-Tasks-3to4  
- **Algorithm**: GRPO (a variant of policy gradient)

Yes, the task is a bit toy-ish—but it captures the essence of R1-Zero: emergent behaviors like self-reflection, verification, backtracking, even language-switching. This setup is ideal for rapid prototyping and experimentation.

### Who is this notebook for?

- Anyone interested in RL training for LLMs  
- Researchers, especially the ones in academia, exploring reasoning in language models

### What should I know before jumping in?

- A working knowledge of the HuggingFace Transformers library  
- Some experience fine-tuning LLMs  
- Familiarity with policy gradient methods (helpful but not required)

## R1-Zero Recipe

The goal is to train a base LLM to **reason** in a way that allows it to **reevaluate** its own outputs and **improve** them, all without human supervision. The DeepSeek R1 paper proposes a surprisingly simple recipe to achieve this, and that's exactly what we'll implement in this notebook.

### The Recipe

Here's the high-level procedure:

1. **Start** with a base LLM and a dataset containing problem prompts paired only with their *final answers* (no intermediate reasoning steps).  
2. For each iteration $i = 0$ to `NUM_ITERATIONS`:
   - Sample a batch of prompts $\{x_i\}_{i=1}^N$ from the dataset.
   - For each prompt, sample $G$ responses from the model:  
     $ y_1, y_2, \cdots, y_G \sim \pi_\theta(y|x) $

     These $G$ responses form what is called a *group* in GRPO.
   - Compute a reward $R_i$ for each response and normalize them tocalculate the GRPO advantage within each group.
   - Create a list of $N \times G$ episodes, i.e., pairs of $(x_i, y_i)$ along with their corresponding advantages.
   - Estimate the policy gradient $\vec{g}_{pg}$ from these episodes.
   - Update the model parameters:  
     $\theta \leftarrow \theta + \eta \vec{g}_{pg}$

### Code Structure Overview

The code you will see is structured directly following this recipe. It boils down to three main components:

1. **Episode Generation**  
   - Generate $ (x, y) $ pairs along with their advantages for each RL iteration.
   
2. **Reward Calculation**  
   - Compute rewards for each generated response.
   
3. **Policy Gradient Estimation**  
   - Use the generated episodes to estimate the policy gradient and perform the model update.

In the end, these three components come together in a simple loop that trains the model, step by step, to develop reasoning capabilities through reinforcement learning.


## Checkpoint Playground

In the `notebooks/checkpoint_playground.ipynb`, you can load the model we already trained with this notebook and interactively test the model's reasoning capabilities. This notebook allows you to input custom prompts and observe the model's responses.

## 1. Imports & DeepSpeed env

In [ ]:
import os
from pathlib import Path

# On Colab, the default HF cache (~/.cache/huggingface) lives on the small home volume.
# Route caches to /content/scratch which is on the large overlay disk.
SCRATCH = Path('/content/scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_home')


In [ ]:
import gc
import re
import time
from typing import Any, Dict, List, Tuple, Union

import deepspeed
import numpy as np
import torch
from datasets import load_dataset
from deepspeed import DeepSpeedEngine
from tqdm import trange
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel
from vllm import LLM, SamplingParams

import wandb
from utils import (
    compute_token_log_probs,
    dump_episodes,
    evaluate_on_test_set,
    find_free_port,
    find_last_checkpoint,
    prepare_model_inputs,
    load_model_into_vllm
)

# Needed to stop DeepSpeed from complaining
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = str(find_free_port())
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"

import ipytest
ipytest.autoconfig()


## 2. Hyperparameters

> A real R1-Zero training run is `NUM_ITERATIONS = 1000` (~12–14 hours on a single A100-80GB). For a quick smoke-test on Colab you can drop it to e.g. 2.

In [ ]:
# Model configuration
MODEL_NAME = "Qwen/Qwen2.5-3B"
MODEL_CHAT_NAME = MODEL_NAME + "-Instruct"

# Dataset configuration
DATASET_NAME = "Jiayi-Pan/Countdown-Tasks-3to4"

# Total number of training iterations
NUM_ITERATIONS = 1000
# Number of episodes to collect per iteration for training
EPISODES_PER_ITERATION = 64
# Number of responses to generate for each input prompt (i.e. group size in GRPO)
GENERATIONS_PER_SAMPLE = 4
# Controls how much the policy can deviate from the reference model
KL_COEFFICIENT = 0.001

# Training hyperparameters
# Batch size for each GPU device during training
PER_DEVICE_BATCH_SIZE = 4
# Learning rate for model updates
LEARNING_RATE = 1e-6

# Sampling parameters
# Maximum number of tokens to generate in each response
MAX_RESPONSE_TOKENS = 1024
# Controls randomness in generation (higher = more random)
TEMPERATURE = 1.0
# Nucleus sampling parameter (1.0 = disabled)
TOP_P = 1.0
# Top-k sampling parameter (-1 = disabled)
TOP_K = -1  # no top k

# DeepSpeed configuration
# DeepSpeed config for the policy model
deepspeed_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {"stage": 2, "overlap_comm": False},
    "train_batch_size": EPISODES_PER_ITERATION,
    "train_micro_batch_size_per_gpu": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": EPISODES_PER_ITERATION // PER_DEVICE_BATCH_SIZE,
    "gradient_clipping": 1.0,
    "optimizer": {
        "type": "AdamW",
        "params": {
            "lr": LEARNING_RATE,
            "betas": (0.9, 0.999),
            "eps": 1e-8,
            "weight_decay": 0.0,
            "torch_adam": True,
        },
    },
}
# DeepSpeed config for the reference model
ref_deepspeed_config = {
    "bf16": {"enabled": True},
    # Note that we don't train the reference model
    # These are just for compatibility with DeepSpeed.
    "train_batch_size": EPISODES_PER_ITERATION,
    "train_micro_batch_size_per_gpu": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": EPISODES_PER_ITERATION // PER_DEVICE_BATCH_SIZE,
}

RUN_NAME = "r1-zero"
EXP_DIR = SCRATCH / "deepseek_r1z_hackathon" / RUN_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Logs and Checkpoints will be saved to: {EXP_DIR}")

## 3. Prompts & Countdown dataset

In [ ]:
SYSTEM_MESSAGE = (
    "You are a helpful assistant. You first think about the reasoning process in the mind "
    "and then provide the user with the answer."
)
PROMPT_TEMPLATE = (
    "Using the numbers {numbers}, create an equation that equals {target}. "
    "You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. "
    "Show your work in <think> </think> tags. And return the final equation and answer in "
    "<answer> </answer> tags, for example <answer>(1 + 2) / (3 * 5)</answer>."
)

In [ ]:
# Load and process dataset
def preprocess_example(example: Dict[str, Any]):
    numbers: List[int] = example["nums"]
    target: int = example["target"]

    prefix = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": PROMPT_TEMPLATE.format(numbers=numbers, target=target)},
        {"role": "assistant", "content": "Let me solve this step by step.\n<think>"},
    ]
    input_ids = tokenizer.apply_chat_template(
        prefix, tokenize=True, continue_final_message=True
    )
    prompt = tokenizer.decode(
        input_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
    )
    return {"prompt": prompt, "input_ids": input_ids}

# Note that the base model and "instruct" model have different eos token. 
# Here we make sure to use the correct one.
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHAT_NAME)
EOS_TOKEN_ID = AutoTokenizer.from_pretrained(MODEL_NAME).eos_token_id
EOS_TOKEN = tokenizer.convert_ids_to_tokens(EOS_TOKEN_ID)

dataset = load_dataset(DATASET_NAME, split="train")
dataset = dataset.map(preprocess_example, num_proc=6)

# Split dataset
train_test_split = dataset.train_test_split(test_size=500, seed=42)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

len(train_dataset), len(test_dataset)

In [ ]:
print("Target: ", train_dataset[0]["target"])
print("Available Numbers: ", train_dataset[0]["nums"])

In [ ]:
print(train_dataset[0]["prompt"])

In [ ]:
print(train_dataset[0]["input_ids"])

## 4. Reward functions

In [ ]:
def format_reward_func(completion: str) -> float:
    """
    Format: <think>...</think>\n</answer>...</answer>

    Also checks that the content within <answer>...</answer> conforms to a
    specified pattern (only digits, + - * / ( ) . and whitespace).

    Args:
        completion (str): Generated output

    Returns:
        float: Reward score
    """
    # Define the allowed pattern (only numbers, +, -, *, /, (, ), ., and whitespace)
    allowed_pattern = r"^[\d+\-*/().\s]+$"

    try:
        # add synthetic <think> as its already part of the prompt and prefilled
        # for the assistant to more easily match the regex
        completion = "<think>" + completion

        # Strip EOS token if present
        if completion.endswith(EOS_TOKEN):
            completion = completion[:-len(EOS_TOKEN)]

        # Check if the format is correct
        # Pattern means:
        # 1) <think>...contents not including other <think> tags...</think>
        # 2) \n
        # 3) <answer>...anything...</answer>
        regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\/think>\n<answer>([\s\S]*?)<\/answer>$"
        match = re.search(regex, completion, re.DOTALL)

        if match is None or len(match.groups()) != 2:
            # Format is incorrect
            return 0.0
        else:
            # Extract the content inside <answer>...</answer>
            answer_content = match.group(2).strip()

            # Check if answer content matches the allowed pattern
            if not re.match(allowed_pattern, answer_content):
                # If it doesn't match, reward is 0.5
                return 0.5
            else:
                # If both format and pattern are correct, reward is 1
                return 1.0
    except Exception:
        # Any error leads to 0 reward
        return 0.0


def equation_reward_func(completion: str, nums: List[int], target: int) -> float:
    """
    Evaluates completion based on mathematical correctness of the answer

    Args:
        completion (str): Generated output
        target (str): Expected answer
        nums (list): Available numbers to use in the equation

    Returns:
        float: Reward score
    """
    try:
        # Check if the format is correct
        match = re.search(r"<answer>(.*?)<\/answer>", completion)
        if match is None:
            return 0.0
        # Extract the "answer" part from the completion
        equation = match.group(1).strip()
        # Extract all numbers from the equation
        used_numbers = [int(n) for n in re.findall(r"\d+", equation)]

        # Check if all numbers are used exactly once
        if sorted(used_numbers) != sorted(nums):
            return 0.0
        # Define a regex pattern that only allows numbers, operators, parentheses, and whitespace
        allowed_pattern = r"^[\d+\-*/().\s]+$"
        if not re.match(allowed_pattern, equation):
            return 0.0

        # Evaluate the equation with restricted globals and locals
        result = eval(equation, {"__builtins__": None}, {})
        # Check if the equation is correct and matches the ground truth
        if abs(float(result) - float(target)) < 1e-5:
            return 1.0
        else:
            return 0.0
    except Exception:
        # If evaluation fails, reward is 0
        return 0.0


In [ ]:
def compute_reward(completion: str, sample: Dict[str, Any]) -> Tuple[float, Dict[str, float]]:
    """Combine the format and equation rewards into a single training signal.

    The two reward components above (format_reward_func, equation_reward_func)
    each return a value in {0.0, 0.5, 1.0}. This function decides HOW to
    combine them into the scalar `reward` the training loop will use.

    Args:
        completion (str): the full generated string for one response.
        sample (dict): the original training sample. Must carry "nums" (list[int])
                       and "target" (int) — the equation reward needs both.

    Returns:
        Tuple[float, Dict[str, float]]:
            - reward: the scalar reward (we use SUM: format_reward + equation_reward,
                      so the range is [0.0, 2.0]). Other choices: weighted sum,
                      gated (equation only counts if format is perfect), max, etc.
            - metrics: per-component breakdown for logging:
                       {"format_reward": ..., "equation_reward": ...}
    """
    nums = sample["nums"]
    target = sample["target"]

    # === STUDENT TODO === #
    raise NotImplementedError("Fill me in!")


In [ ]:
%%ipytest -q

def test_compute_reward_full():
    """Perfect format + correct equation = 2.0 (1.0 + 1.0)."""
    sample = {"nums": [1, 2], "target": 3, "input_ids": [0]}
    completion = "I think.</think>\n<answer>1+2</answer>"
    r, m = compute_reward(completion, sample)
    assert r == 2.0, r
    assert m["format_reward"] == 1.0
    assert m["equation_reward"] == 1.0

def test_compute_reward_format_only_wrong_eqn():
    """Perfect format, wrong equation = 1.0 (1.0 + 0.0)."""
    sample = {"nums": [1, 2], "target": 99, "input_ids": [0]}
    completion = "I think.</think>\n<answer>1+2</answer>"
    r, m = compute_reward(completion, sample)
    assert r == 1.0, r
    assert m["format_reward"] == 1.0
    assert m["equation_reward"] == 0.0

def test_compute_reward_garbage():
    """Bad format, no answer tag = 0.0."""
    sample = {"nums": [1, 2], "target": 3, "input_ids": [0]}
    r, m = compute_reward("Hello world", sample)
    assert r == 0.0, r
    assert m["format_reward"] == 0.0
    assert m["equation_reward"] == 0.0


## 5. Episode generation

In [ ]:
def compute_group_relative_advantages(rewards: np.ndarray) -> np.ndarray:
    """GRPO group-relative advantage: subtract group mean, divide by group std (+eps).

    GRPO replaces the value-network baseline with a *group baseline*: for each
    prompt we sample G responses, score them all, then center & rescale by the
    group's own statistics. No critic needed.

    Shapes:
        rewards : (G,) np.float64 — the G raw rewards from one prompt's group
        returns : (G,) np.float64 — normalized advantages, group-mean ≈ 0

    Edge case:
        If all G rewards are equal (std == 0), the eps in the denominator
        keeps the result finite — all advantages collapse to zero.

    Formula:
        A_i = (r_i - mean(r)) / (std(r) + eps),  with eps = 1e-4
    """
    # === STUDENT TODO === #
    raise NotImplementedError("Fill me in!")


In [ ]:
%%ipytest -q

def test_grpo_adv_uniform_rewards_zero():
    """All-equal rewards → all-zero advantages (mean = r, std ≈ 0)."""
    r = np.array([0.5, 0.5, 0.5, 0.5])
    a = compute_group_relative_advantages(r)
    assert np.allclose(a, 0.0, atol=1e-3), a

def test_grpo_adv_centering_and_sign():
    """Hand-computed normalization with a non-degenerate group."""
    r = np.array([0.0, 0.0, 1.0, 1.0])
    a = compute_group_relative_advantages(r)
    # mean=0.5, std=0.5, eps=1e-4 → a ≈ ±1.0
    expected = (r - r.mean()) / (r.std() + 1e-4)
    assert np.allclose(a, expected), (a, expected)
    assert a[0] < 0 and a[2] > 0, a

def test_grpo_adv_shape_preserved():
    r = np.zeros(8, dtype=np.float64)
    assert compute_group_relative_advantages(r).shape == (8,)


In [ ]:
def create_training_episodes(
    samples: List[Dict[str, Any]],
    all_generations: List[List[int]],
    all_finish_reasons: List[str],
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Process model generations and calculate rewards for training episodes.

    This function processes generated responses and calculates rewards for training episodes by:
    1. Grouping generations by sample (GENERATIONS_PER_SAMPLE responses per input)
    2. Computing rewards and advantages for each response (uses `compute_reward` +
       `compute_group_relative_advantages`)
    3. Broadcasting the per-response advantage to a per-token list so the
       training loop can index it the same way as input/response token ids.

    Args:
        samples: List of input samples, each containing:
            - input_ids: List[int], tokenized input prompt
            - nums: List[int], numbers to use in equation
            - target: int, target value for equation
        all_generations: List of token ID sequences for each generated response.
                         Length is len(samples) * GENERATIONS_PER_SAMPLE.
        all_finish_reasons: List of finish reasons for each generation ("stop" or other)

    Returns:
        Tuple containing:
        1. episodes dict (training data):
            - all_query_token_ids: List[List[int]], input token IDs repeated for each generation
            - all_response_token_ids: List[List[int]], response token IDs (NOT padded)
            - all_advantages: List[List[float]], per-token advantages (one float per response token)
        2. stats dict (per-iteration logging):
            - response_lengths: List[int], lengths of generated responses
            - rewards: List[float], raw reward values
            - non_stop_rate: List[bool], whether each generation ended for non-"stop" reasons
            - reward_metrics/*: Various reward component metrics

    Example:
        >>> samples = [{"input_ids": [1,2,3], "nums": [1,2,3], "target": 6}]
        >>> generations = [[4,5, EOS_TOKEN_ID], [6,7], [8,9, EOS_TOKEN_ID]]  # 3 generations per sample
        >>> finish_reasons = ["stop", "length", "stop"]
        >>> episodes, stats = create_training_episodes(samples, generations, finish_reasons)
        >>> episodes
        {
            'all_query_token_ids': [[1,2,3], [1,2,3], [1,2,3]],
            'all_response_token_ids': [[4,5,EOS_TOKEN_ID], [6,7], [8,9,EOS_TOKEN_ID]],
            'all_advantages': [[0.5,0.5,0.5], [-1.0,-1.0], [0.5,0.5,0.5]]
        }
    """
    assert len(all_generations) == len(all_finish_reasons)
    assert len(all_generations) == len(samples) * GENERATIONS_PER_SAMPLE

    # Group generation indices by their source sample.
    # Each sample contributes GENERATIONS_PER_SAMPLE consecutive generations.
    groups = [
        list(range(i, i + GENERATIONS_PER_SAMPLE))
        for i in range(0, len(all_generations), GENERATIONS_PER_SAMPLE)
    ]  # example: [[0, 1, 2, 3], [4, 5, 6, 7], ...]

    all_query_token_ids: List[List[int]] = []
    all_responses_token_ids: List[List[int]] = []
    all_advantages: List[List[float]] = []

    stats: Dict[str, Any] = {
        "response_lengths": [],
        "rewards": [],
        "non_stop_rate": [],
    }

    # === STUDENT TODO === #
    raise NotImplementedError("Fill me in!")

    episodes = {
        "all_query_token_ids": all_query_token_ids,
        "all_response_token_ids": all_responses_token_ids,
        "all_advantages": all_advantages,
    }

    return episodes, stats


In [ ]:
%%ipytest -q

def test_create_training_episodes_shapes_and_stats():
    """Smoke test on a 1-sample, 4-generation case.

    Note: GENERATIONS_PER_SAMPLE must == 4 for this fixture; tokenizer.batch_decode
    is called with random integer ids so the resulting strings won't match any
    reward regex → all rewards 0.0 → all advantages 0.0 (eps clamp).
    """
    case = {
        "sample": {"input_ids": [1, 2, 3], "nums": [1, 2, 3], "target": 6},
        "generations": [[4, 5, 22, 33], [6, 7], [8, 9, 11], [10, 11]],
        "finish_reasons": ["stop", "length", "stop", "stop"],
    }
    assert GENERATIONS_PER_SAMPLE == 4, "fixture assumes 4-generation groups"

    episodes, stats = create_training_episodes(
        [case["sample"]], case["generations"], case["finish_reasons"],
    )

    # query token ids: input_ids repeated 4× (one per generation)
    assert episodes["all_query_token_ids"] == [[1, 2, 3]] * 4

    # response token ids preserved 1:1
    assert episodes["all_response_token_ids"] == case["generations"]

    # per-token advantage list lengths match response lengths
    assert [len(a) for a in episodes["all_advantages"]] == [4, 2, 3, 2]

    # All rewards are 0.0 (random ints don't decode to a valid format) →
    # all-equal rewards → all-zero advantages (within eps).
    flat = [v for row in episodes["all_advantages"] for v in row]
    assert all(abs(v) < 1e-2 for v in flat), flat

    # stats sanity
    assert len(stats["rewards"]) == 4
    assert stats["non_stop_rate"] == [False, True, False, False]
    assert stats["response_lengths"] == [4, 2, 3, 2]


## 6. Policy-gradient loss

> **Colab fix:** the upstream `utils.py` (v0.2) was updated to require a separate `labels_mask` field, while this notebook was authored against the older `labels[-100]`-only API. The cell below forwards `labels_mask` from the prepared batch into `compute_token_log_probs` so the two stay in sync.

In [ ]:
def compute_kl_penalty(
    logps: torch.Tensor,
    ref_logps: torch.Tensor,
    shift_labels_mask: torch.Tensor,
) -> torch.Tensor:
    """K3 KL approximator (Schulman): exp(d) - d - 1, where d = ref - log.

    This is an unbiased low-variance estimator of KL(π || π_ref) per token.
    See http://joschu.net/blog/kl-approx.html for the derivation.

    Shapes:
        logps             : (B, T-1) float — log π_θ for each response token
        ref_logps         : (B, T-1) float — log π_ref for the same tokens
        shift_labels_mask : (B, T-1) float — 1.0 on response tokens, 0.0 on prompt/pad

    Returns:
        kl : (B, T-1) float — per-token KL contribution, ALREADY MASKED
             (positions where shift_labels_mask == 0 are zeroed out).

    Properties to remember:
        * KL(P || P) == 0  (returns zero when logps == ref_logps)
        * KL >= 0 always (the K3 estimator is non-negative pointwise)
    """
    # === STUDENT TODO === #
    raise NotImplementedError("Fill me in!")


In [ ]:
%%ipytest -q

def test_kl_zero_when_logps_match():
    """π == π_ref → KL == 0."""
    lp = torch.zeros(2, 5)
    ref = lp.clone()
    mask = torch.ones(2, 5)
    kl = compute_kl_penalty(lp, ref, mask)
    assert torch.allclose(kl, torch.zeros_like(kl), atol=1e-6)

def test_kl_nonnegative():
    """K3 KL approximator is non-negative pointwise."""
    torch.manual_seed(0)
    lp  = torch.randn(2, 5) * 0.5
    ref = torch.randn(2, 5) * 0.5
    mask = torch.ones(2, 5)
    kl = compute_kl_penalty(lp, ref, mask)
    assert (kl >= -1e-6).all(), kl.min()

def test_kl_mask_zeroes_positions():
    """Masked-out positions contribute 0, regardless of logp difference."""
    lp  = torch.tensor([[0.0, 0.0, 0.0]])
    ref = torch.tensor([[1.0, 2.0, 3.0]])  # large divergence on every position
    mask = torch.tensor([[0.0, 0.0, 0.0]])
    kl = compute_kl_penalty(lp, ref, mask)
    assert torch.allclose(kl, torch.zeros_like(lp))


In [ ]:
def compute_policy_loss_term(
    logps: torch.Tensor,
    advantages_shifted: torch.Tensor,
    shift_labels_mask: torch.Tensor,
) -> torch.Tensor:
    """The REINFORCE-style policy loss term: − log π · A · mask.

    Shapes:
        logps              : (B, T-1) float — log π_θ for each response token
        advantages_shifted : (B, T-1) float — advantages already shifted to
                              align with the next-token log-prob (advantages[..., 1:])
        shift_labels_mask  : (B, T-1) float — 1.0 on response tokens, 0.0 elsewhere

    Returns:
        policy_loss : (B, T-1) float — per-token policy-loss contribution,
                      already MASKED. The training loop sums this and divides
                      by total_response_len.

    Sign convention:
        policy_loss = − A · log π · mask
        Positive A + positive log π → negative loss term (we WANT to push log π up).
        Gradient ascent on log π is gradient descent on this term.
    """
    # === STUDENT TODO === #
    raise NotImplementedError("Fill me in!")


In [ ]:
%%ipytest -q

def test_pg_zero_advantages_zero_loss():
    """A == 0 everywhere → loss term is zero."""
    lp = torch.randn(2, 5)
    adv = torch.zeros(2, 5)
    mask = torch.ones(2, 5)
    pl = compute_policy_loss_term(lp, adv, mask)
    assert torch.allclose(pl, torch.zeros_like(pl))

def test_pg_sign_correct():
    """A>0 + log π>0 → loss < 0 (we DECREASE loss by pushing log π up)."""
    lp  = torch.tensor([[0.5]])
    adv = torch.tensor([[1.0]])
    mask = torch.tensor([[1.0]])
    pl = compute_policy_loss_term(lp, adv, mask)
    # -0.5 * 1.0 * 1.0 = -0.5
    assert torch.allclose(pl, torch.tensor([[-0.5]])), pl

def test_pg_mask_kills_position():
    """mask==0 zeros the contribution regardless of log π and A."""
    lp  = torch.tensor([[0.5]])
    adv = torch.tensor([[1.0]])
    mask = torch.tensor([[0.0]])
    pl = compute_policy_loss_term(lp, adv, mask)
    assert pl.abs().item() < 1e-6, pl


In [ ]:
def compute_entropy(
    logps: torch.Tensor,
    shift_labels_mask: torch.Tensor,
) -> torch.Tensor:
    """Per-alive-token entropy proxy.

    A scalar mean over alive (response) tokens of − log π. This is a
    *proxy* for entropy (the true H(π) would average over the action
    distribution at each step, not over the sampled token). It's good
    enough for monitoring policy collapse / over-confidence during training.

    Shapes:
        logps             : (B, T-1) float — log π_θ for each response token
        shift_labels_mask : (B, T-1) float — 1.0 alive, 0.0 dead

    Returns:
        scalar tensor (0-dim) — mean(− log π) over the alive tokens.

    Edge case:
        If all positions are masked, divide by max(sum_mask, 1) to avoid
        division by zero.
    """
    # === STUDENT TODO === #
    raise NotImplementedError("Fill me in!")


In [ ]:
%%ipytest -q

def test_entropy_uniform_logps():
    """Uniform log π → entropy = − log π everywhere."""
    lp = torch.full((2, 4), -1.5)
    mask = torch.ones(2, 4)
    ent = compute_entropy(lp, mask)
    # masked-mean of (-(-1.5)) = 1.5
    assert torch.allclose(ent, torch.tensor(1.5), atol=1e-6), ent

def test_entropy_mask_aware():
    """Dead tokens are excluded from BOTH numerator and denominator."""
    lp = torch.tensor([[-1.0, -1.0, -100.0, -100.0]])  # last 2 are "dead"
    mask = torch.tensor([[1.0, 1.0, 0.0, 0.0]])
    ent = compute_entropy(lp, mask)
    # Only first two positions count: mean(-(-1.0), -(-1.0)) = 1.0
    assert torch.allclose(ent, torch.tensor(1.0), atol=1e-6), ent

def test_entropy_all_dead_safe():
    """All-dead mask is safe (no NaN)."""
    lp = torch.full((1, 3), -2.0)
    mask = torch.zeros(1, 3)
    ent = compute_entropy(lp, mask)
    assert torch.isfinite(ent).item(), ent


In [ ]:
def compute_pg_loss_inner(
    logps: torch.Tensor,
    ref_logps: torch.Tensor,
    advantages: torch.Tensor,
    labels_mask: torch.Tensor,
    kl_coeff: float,
    total_response_len: int,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Glue: assemble the full PG loss from the three sub-pieces above.

    Pure-tensor — no model forward passes. Stitched here so `compute_pg_loss`
    only has to handle the model.forward orchestration.

    Shapes:
        logps              : (B, T-1) float — already shifted (model gave T outputs, we kept [..., :-1])
        ref_logps          : (B, T-1) float
        advantages         : (B, T)   float — broadcast over the FULL sequence; we shift to (B, T-1)
        labels_mask        : (B, T)   float/bool — 1 on response tokens of the FULL sequence
        kl_coeff           : float — KL_COEFFICIENT
        total_response_len : int — divisor used for both loss and metrics (counted across the whole batch)

    Returns:
        loss    : 0-d tensor — (policy_loss + kl_coeff * kl_penalty).sum() / total_response_len
        metrics : dict with per-component scalar floats: policy_loss, kl_penalty, entropy
    """
    shift_labels_mask = labels_mask[..., 1:].float()  # (B, T-1)

    kl_penalty       = compute_kl_penalty(logps, ref_logps, shift_labels_mask)              # (B, T-1)
    policy_loss_term = compute_policy_loss_term(logps, advantages[..., 1:], shift_labels_mask)  # (B, T-1)
    entropy_scalar   = compute_entropy(logps, shift_labels_mask)                            # ()

    loss = (policy_loss_term + kl_coeff * kl_penalty).sum() / total_response_len

    metrics = {
        "policy_loss": (policy_loss_term.sum() / total_response_len).item(),
        "kl_penalty":  (kl_penalty.sum() / total_response_len).item(),
        "entropy":     entropy_scalar.item(),
    }
    return loss, metrics


In [ ]:
def compute_pg_loss(
    policy_model: Union[DeepSpeedEngine, PreTrainedModel],
    reference_model: Union[DeepSpeedEngine, PreTrainedModel],
    batch: Dict[str, torch.Tensor],
    total_response_len: int,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    """Outer loss: run both model forwards, hand off to compute_pg_loss_inner.

    This wrapper exists so the heavy GPU code (DeepSpeed forwards) stays
    isolated from the pure-tensor math (KL/PG/entropy). The inner function
    is unit-tested on CPU; this outer one is exercised end-to-end by the
    training loop on Colab A100.
    """
    model_inputs = {
        "input_ids":      batch["input_ids"],
        "attention_mask": batch["attention_mask"],
        "labels":         batch["labels"],
        "labels_mask":    batch["labels_mask"],   # required by utils.py v0.2
    }

    with torch.no_grad():
        ref_logps = compute_token_log_probs(reference_model, model_inputs, TEMPERATURE)

    logps = compute_token_log_probs(policy_model, model_inputs, TEMPERATURE)

    return compute_pg_loss_inner(
        logps=logps,
        ref_logps=ref_logps,
        advantages=batch["advantages"],
        labels_mask=batch["labels_mask"],
        kl_coeff=KL_COEFFICIENT,
        total_response_len=total_response_len,
    )


## 7. Initialize policy + reference models, vLLM engine, wandb

In [ ]:
# Initialize main and reference models
policy_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    device_map=0,
)
reference_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    device_map=0,
)
policy_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})


# Initialize DeepSpeed engines
policy_model, *_ = deepspeed.initialize(
    model=policy_model,
    config=deepspeed_config,
    model_parameters=policy_model.parameters(),
)
reference_model, *_ = deepspeed.initialize(
    model=reference_model,
    config=ref_deepspeed_config,
)

reference_model.module.cpu()

############################################
# Initialize vLLM (Inference) engine
############################################

inference_engine = LLM(
    model=MODEL_NAME,
    skip_tokenizer_init=False,
    gpu_memory_utilization=0.2,
    enable_prefix_caching=True,
    swap_space=1,
    scheduling_policy="fcfs",
    dtype=torch.bfloat16,
    max_model_len=2048,
    enable_sleep_mode=True,
)

# Wandb for logging
wandb.init(
    project="r1-aha-moment",
    name=RUN_NAME,
    config={
        "model_name": MODEL_NAME,
        "learning_rate": LEARNING_RATE,
        "num_iterations": NUM_ITERATIONS,
        "episodes_per_iteration": EPISODES_PER_ITERATION,
        "rollouts_per_episode": GENERATIONS_PER_SAMPLE,
        "kl_coefficient": KL_COEFFICIENT,
        "temperature": TEMPERATURE,
    },
)

# Load checkpoint if it exists
begin_iter = 0
ckpt_path, ckpt_iter = find_last_checkpoint(EXP_DIR)
if ckpt_path is not None:
    print(f"Resuming from checkpoint {ckpt_path} at iteration {ckpt_iter}")
    out = policy_model.load_checkpoint(ckpt_path / "deepspeed")
    if out is None:
        raise RuntimeError(f"Failed to load checkpoint {ckpt_path}")
    begin_iter = ckpt_iter + 1
    load_model_into_vllm(policy_model, inference_engine)

## 8. Training loop

> **Colab change:** the periodic evaluation block (`if iteration % 25 == 0`) is commented out by default — re-enable it for a real training run if you want eval-set metrics every 25 iters.

In [ ]:
for iteration in trange(NUM_ITERATIONS):
    print(f"Iteration {iteration}/{NUM_ITERATIONS}")

    metrics = {}

    #########################################################
    # Evaluation
    #########################################################

    eval_stats = None
    # Evaluation disabled on Colab smoke test. Re-enable for a real run:
    #   if iteration % 25 == 0:
    #       eval_episodes, eval_stats = evaluate_on_test_set(...)
    #       ...


    #########################################################
    # Generate Episodes
    #########################################################

    # Sample training batch
    num_samples = EPISODES_PER_ITERATION // GENERATIONS_PER_SAMPLE
    indices = np.random.choice(
        len(train_dataset), size=num_samples, replace=False
    )
    samples = train_dataset.select(indices)

    # Sample responses
    outputs = inference_engine.generate(
        prompt_token_ids=samples["input_ids"],
        sampling_params=SamplingParams(
            n=GENERATIONS_PER_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            max_tokens=MAX_RESPONSE_TOKENS,
            detokenize=False,
            stop_token_ids=[EOS_TOKEN_ID],
        )
    )
    all_generations = [list(g.token_ids) for out in outputs for g in out.outputs]
    all_finish_reasons = [g.finish_reason for out in outputs for g in out.outputs]
    inference_engine.sleep(1)

    print(f"Generated {len(all_generations)} responses")
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)

    # Process responses and calculate rewards
    episodes, episodes_stats = create_training_episodes(
        samples,
        all_generations,
        all_finish_reasons,
    )
    for k, v in episodes_stats.items():
        metrics.setdefault(k, []).extend(v)

    episode_table = dump_episodes(
        episodes=episodes,
        episodes_stats=episodes_stats,
        exp_dir=EXP_DIR,
        tokenizer=tokenizer,
        iteration=iteration,
    )

    #########################################################
    # Training
    #########################################################

    # Prepare training batch
    model_inputs = prepare_model_inputs(
        query_token_ids=episodes["all_query_token_ids"],
        response_token_ids=episodes["all_response_token_ids"],
        advantages=episodes["all_advantages"],
        device="cuda"
    )

    # Calculate losses and update model
    policy_model.train()
    reference_model.module.cuda()
    reference_model.eval()

    total_response_len = (model_inputs["labels"] != -100).sum().item()

    for i in trange(0, EPISODES_PER_ITERATION, PER_DEVICE_BATCH_SIZE, desc="Gradient Accumulation"):
        batch = {
            k: v[i : i + PER_DEVICE_BATCH_SIZE]
            for k, v in model_inputs.items()
        }

        # Compute policy gradient loss
        loss, loss_metrics = compute_pg_loss(
            policy_model=policy_model,
            reference_model=reference_model,
            batch=batch,
            total_response_len=total_response_len,
        )

        # Track metrics
        metrics.setdefault("loss", []).append(loss.item())
        grad_norm = policy_model.get_global_grad_norm()
        if grad_norm is not None:
            grad_norm = grad_norm.item()
        metrics.setdefault("grad_norm", []).append(grad_norm)
        for k, v in loss_metrics.items():
            metrics.setdefault(k, []).append(v.item() if isinstance(v, torch.Tensor) else v)

        # Backpropagation and optimization step
        policy_model.backward(loss, scale_wrt_gas=False)
        
        # Free memory
        del loss, loss_metrics
        if policy_model.is_gradient_accumulation_boundary():
            reference_model.module.cpu()

        policy_model.step()

    #########################################################
    # Update inference engine weights
    #########################################################
    
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)

    inference_engine.wake_up()
    load_model_into_vllm(policy_model, inference_engine)

    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)


    #########################################################
    # Log metrics
    #########################################################

    train_metrics = {
        k: np.mean(v) for k, v in metrics.items() if None not in v
    }
    train_metrics["learning_rate"] = policy_model.get_lr()[0]
    logs = {
        "iteration": iteration,
        f"episodes/iter_{iteration:06d}": episode_table,
        **{f"train/{k}": v for k, v in train_metrics.items()},
    }
    if eval_stats is not None:
        eval_metrics = {k: np.mean(v) for k, v in eval_stats.items() if None not in v}
        logs.update({f"eval/{k}": v for k, v in eval_metrics.items()})
    wandb.log(logs)

    selected_keys = [
        "train/kl_penalty",
        "train/rewards",
        "train/reward_metrics/format_reward",
        "train/reward_metrics/equation_reward",
        "eval/rewards",
        "eval/reward_metrics/format_reward",
        "eval/reward_metrics/equation_reward",
    ]
    selected_metrics = {k: logs[k] for k in selected_keys if k in logs}
    print(f"KEY METRICS: {selected_metrics}")

    if iteration % 50 == 0 and iteration != 0:
        policy_model.module.save_pretrained(
            str(EXP_DIR / "checkpoints" / f"ckpt_{iteration:06d}" / "hf_model")
        )
        policy_model.save_checkpoint(
            str(EXP_DIR / "checkpoints" / f"ckpt_{iteration:06d}" / "deepspeed")
        )

## 9. Checkpoint Playground

Port of [`notebooks/checkpoint_playground.ipynb`](https://github.com/McGill-NLP/nano-aha-moment/blob/main/notebooks/checkpoint_playground.ipynb).

Loads the **published 1000-iter trained checkpoint** [`McGill-NLP/nano-aha-moment-3b`](https://huggingface.co/McGill-NLP/nano-aha-moment-3b) and runs two demos: a Countdown problem and a general chat prompt.

### Two intended paths through this notebook

| Path | What you run |
|------|--------------|
| **Training** | Sections 0 → 8 (skip 9). Trains your own policy from the Qwen2.5-3B base. |
| **Inference-only** | Sections 0 (setup) → 9 (playground). Skips the `policy_model`, `reference_model`, and training-time `inference_engine` init. |

Don't run both in the same session — the policy + reference + training-time vLLM together with a second playground vLLM won't fit on a single A100-80GB.

> Swap `CHECKPOINT_OR_NAME` below if you want to play with a different model (`Qwen/Qwen2.5-3B`, your own `EXP_DIR/checkpoints/...`, etc.).

In [ ]:
import os
from pathlib import Path

# Use the same scratch dir as the training section so we share the HF cache.
SCRATCH = Path('/content/scratch')
SCRATCH.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_home')
os.environ['VLLM_USE_V1'] = '0'

from transformers import AutoTokenizer

CHECKPOINT_OR_NAME = 'McGill-NLP/nano-aha-moment-3b'
# Other options to try:
# CHECKPOINT_OR_NAME = 'Qwen/Qwen2.5-3B'
# CHECKPOINT_OR_NAME = str(EXP_DIR / 'checkpoints' / 'ckpt_000050' / 'hf_model')   # your own
CHAT_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # tokenizer the checkpoint was trained with

tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL_NAME)
print("Tokenizer loaded:", CHAT_MODEL_NAME)
print("Checkpoint to load:", CHECKPOINT_OR_NAME)

In [ ]:
import torch
from vllm import LLM, SamplingParams

inference_engine = LLM(
    model=CHECKPOINT_OR_NAME,
    gpu_memory_utilization=0.5,
    dtype=torch.bfloat16,
    swap_space=2,
    enable_prefix_caching=True,
    max_model_len=2048,
    max_seq_len_to_capture=2048,
)

In [ ]:
def format_response(query, response):
    from IPython.display import HTML

    # Escape <think> </think> <answer> </answer> and any HTML tags
    response = response.replace("<", "&lt;").replace(">", "&gt;")
    query = query.replace("<", "&lt;").replace(">", "&gt;")

    formatted_html = f"""
    <div style="background-color: #f8f9fa; padding: 15px; border-radius: 5px; border: 1px solid #ddd;">
        <h3 style="color: #333; margin-top: 0;">Query:</h3>
        <pre style="background-color: #e9f7fe; padding: 10px; border-radius: 3px; overflow-x: auto; white-space: pre-wrap; word-wrap: break-word; color: #0066cc;">{query}</pre>
        <h3 style="color: #333; margin-top: 10px;">Generated Response:</h3>
        <pre style="background-color: #f5f5f5; padding: 10px; border-radius: 3px; overflow-x: auto; white-space: pre-wrap; word-wrap: break-word; color: #333333;">{response}</pre>
    </div>
    """
    return HTML(formatted_html)


def generate_chat_prompt(query, assistance_prefix="Let me think step by step\n<think>"):
    SYSTEM_MESSAGE = (
        "You are a helpful assistant. You first think about the reasoning process in the mind "
        "and then provides the user with the answer."
    )
    r1_prefix = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": f"{query}"},
    ]
    if assistance_prefix is not None:
        r1_prefix.append({"role": "assistant", "content": assistance_prefix})

    input_ids = tokenizer.apply_chat_template(
        r1_prefix, tokenize=True, continue_final_message=assistance_prefix is not None
    )
    prompt = tokenizer.decode(
        input_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False
    )
    return {"prompt": prompt, "input_ids": input_ids}


def preprocess_countdown_example(example):
    SYSTEM_MESSAGE = (
        "You are a helpful assistant. You first think about the reasoning process in the mind "
        "and then provides the user with the answer."
    )
    PROMPT_TEMPLATE = (
        "Using the numbers {numbers}, create an equation that equals {target}. "
        "You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. "
        "Show your work in <think> </think> tags. And return the final equation and answer in "
        "<answer> </answer> tags, for example <answer>(1 + 2) / (3 * 5)</answer>."
    )
    numbers = example["nums"]
    target = example["target"]

    chat_messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": PROMPT_TEMPLATE.format(numbers=numbers, target=target)},
        {"role": "assistant", "content": "Let me think step by step\n<think>"},
    ]

    input_ids = tokenizer.apply_chat_template(
        chat_messages, tokenize=True, continue_final_message=True
    )
    prompt = tokenizer.decode(input_ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    return {"input_ids": input_ids, "prompt": prompt}

### Countdown

In [ ]:
sample = {"nums": [7, 71, 19, 4], "target": 68}
sample.update(preprocess_countdown_example(sample))

print(f"######################## Prompt:\n`{sample['prompt']}`")

generation = inference_engine.generate(
    prompt_token_ids=sample["input_ids"],
    sampling_params=SamplingParams(
        temperature=0.3,
        max_tokens=1024,
        top_p=1.0,
        n=1,
    ),
)
response = tokenizer.decode(generation[0].outputs[0].token_ids)
format_response(sample["prompt"], response)

### General

In [ ]:
o_1 = """A quadratic equation has roots that are also the solutions to the system:

\\[
\\begin{cases}
x + y = 7 \\\\
xy = 10
\\end{cases}
\\]

1. Find the quadratic equation whose roots are \\( x \\) and \\( y \\).
2. Solve the quadratic equation.
3. Verify that the roots satisfy the original system.
"""

o = """Hello, how are you?"""

o_3 = """My slurm job failed. I look at the stdout and I observe this:

### stdout ###
slurmstepd: error: container_p_join: open failed for /var/opt/slurm/localstorage/6450599/.ns: No such file or directory
slurmstepd: error: container_g_join(6450599): No such file or directory"""

o_4 = """
How many of the first 500 positive integers are divisible by 3, 4 and 5?
"""

o_5 = """
You have 1,2,3,4. Provide a math equation that equals 10. You can use each number only once. You can use basic arithmetic operations (+, -, *, /).
"""

sample = generate_chat_prompt(
    f"{o}\n"
    "Show your work in <think> </think> tags. And return the final answer in "
    "<answer> </answer> tags"
)

print(f"######################## Prompt:\n`{sample['prompt']}`")

generation = inference_engine.generate(
    prompt_token_ids=sample["input_ids"],
    sampling_params=SamplingParams(
        temperature=0.6,
        max_tokens=1024,
        top_p=1.0,
        n=1,
    ),
)
response = tokenizer.decode(generation[0].outputs[0].token_ids)
format_response(o, response)